# Retrieval-Augmented Generation (RAG) Pipeline for Technical PDF Documents

This notebook demonstrates how a Retrieval-Augmented Generation (RAG) pipeline can be used to retrieve information from technical PDF documents.

The objective is to extract text from multiple PDF files, clean and structure the extracted content, and prepare it for later use in an AI retrieval system.

In this step, we extract text from all PDF files located in the `data` folder and preprocess the content to make it suitable for later retrieval.

A helper function called 'clean_text()` is defined to preprocess the extracted text. This function removes unwanted elements commonly found in technical documents, including:
- citation markers such as `[1]` or `[23]`
- URLs
- excessive whitespace

Next, the script locates the folder containing the PDF documents and iterates through all files with the `.pdf` extension.

The script then processes all PDF documents located in the specified data folder. It iterates through each `.pdf` file, opens the document, and extracts the text page by page. The extracted text is split into paragraphs using regular expressions, after which each paragraph is cleaned using the `clean_text()` function. Very small text fragments (less than 50 characters) are discarded to remove incomplete or noisy content, ensuring that only meaningful text segments are stored.

In [1]:
import pdfplumber
import re
from pathlib import Path

def clean_text(text: str) -> str:
    # Remove citation brackets like [1], [a], [23]
    text = re.sub(r"\[[^\]]*\]", "", text)
    
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    
    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

path = "data"
file_path = Path(path)
BASE_DIR = Path().resolve().parent
pdf_folder = BASE_DIR / file_path
all_paragraphs = []
for pdf_file in pdf_folder.glob("*.pdf"):
    team_name = pdf_file.stem  # e.g., devops, docker
    with pdfplumber.open(pdf_file) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                raw_paragraphs = re.split(r'\n\s*\n|[-]{5,}', text)
                for p in raw_paragraphs:
                    clean_p = clean_text(p.strip())
                    if clean_p and len(clean_p) > 50:  # ignore tiny fragments
                        all_paragraphs.append({
                            "team": team_name,
                            "text": clean_p
                    })

print(all_paragraphs[:1])

[{'team': 'DevOps', 'text': 'DevOps DevOps is the integration and automation of software development and information technology operations. DevOps encompasses necessary tasks of software development and can lead to shortening development time and improving the development life cycle. According to Neal Ford, DevOps, particularly through continuous delivery, employs the "Bring the pain forward" principle, tackling tough tasks early, fostering automation and swift issue detection. Software programmers and architects should use fitness functions to keep their software in check. Although debated, DevOps is characterized by key principles: shared ownership, workflow automation, and rapid feedback. From an academic perspective, Len Bass, Ingo Weber, and Liming Zhu —three computer science researchers from the CSIRO and the Software Engineering Institute— suggested defining DevOps as "a set of practices intended to reduce the time between committing a change to a system and the change being pla

## Text Chunking for Retrieval

After extracting and cleaning the paragraphs, the text is further divided into smaller segments called **chunks**. Chunking is an important step in a RAG pipeline because large blocks of text are difficult for retrieval systems and embedding models to process efficiently.

In this step, each paragraph is first split into individual sentences using the `sent_tokenize` function from the `nltk` library. Sentences are then grouped together until a predefined word limit (`max_words`) is reached. This ensures that each chunk remains within a manageable size while still preserving meaningful context.

To avoid losing contextual continuity between chunks, a small overlap of sentences (`overlap_sentences`) is kept between consecutive chunks. Each generated chunk is stored together with metadata such as a unique `chunk_id`, the source document (`team`), and the original paragraph index.

Finally, the script outputs the total number of generated chunks, which will later be used for embedding and retrieval in the RAG pipeline.

In [2]:
from nltk.tokenize import sent_tokenize

max_words=250 
overlap_sentences=1
chunks = []
chunk_id = 0

for p_index, item in enumerate(all_paragraphs):
    text = item["text"]
    team = item["team"]

    sentences = sent_tokenize(text)

    current_chunk = []
    current_word_count = 0

    for i, sentence in enumerate(sentences):
        sentence_word_count = len(sentence.split())

        if current_word_count + sentence_word_count <= max_words:
            current_chunk.append(sentence)
            current_word_count += sentence_word_count
        else:
            # Save chunk
            chunks.append({
                "chunk_id": chunk_id,
                "team": team,
                "paragraph_index": p_index,
                "text": " ".join(current_chunk)
            })
            chunk_id += 1

            # Start new chunk with overlap
            overlap = current_chunk[-overlap_sentences:] if overlap_sentences > 0 else []
            current_chunk = overlap + [sentence]
            current_word_count = sum(len(s.split()) for s in current_chunk)

    # Save remaining chunk
    if current_chunk:
        chunks.append({
            "chunk_id": chunk_id,
            "team": team,
            "paragraph_index": p_index,
            "text": " ".join(current_chunk)
        })
        chunk_id += 1

print(f"Generated {len(chunks)} chunks from {len(all_paragraphs)} paragraphs")

Generated 74 chunks from 37 paragraphs


## Generating Embeddings and Storing Chunks in a Vector Database

In this step, the text chunks generated earlier are converted into **vector embeddings** and stored in a **vector database** for efficient semantic retrieval.

First, the required libraries are imported: `chromadb` for managing the vector database and the OpenAI client for generating embeddings. The function `get_embedding()` uses the OpenAI embedding model `text-embedding-3-small` to transform each text chunk into a numerical vector representation that captures its semantic meaning.

A persistent Chroma database is then initialized, and a collection named `technical_docs` is created (or loaded if it already exists). The script iterates through all generated chunks, computes an embedding for each chunk, and stores it in the collection along with the original text and metadata such as the document source (`team`) and paragraph index.

Storing these embeddings in a vector database allows the system to later perform **semantic search**, retrieving the most relevant text chunks based on the similarity between embeddings rather than simple keyword matching.

In [4]:
import chromadb
from openai import OpenAI
import os

client_openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def get_embedding(text: str) -> list:
    """
    Generate embedding for a single text chunk using OpenAI text-embedding-3-small.
    """
    response = client_openai.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

CHROMA_COLLECTION = "technical_docs"
DATA_DIR = BASE_DIR / "data"
CHROMA_DIR = DATA_DIR / "chroma_db"

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

collection = chroma_client.get_or_create_collection(name=CHROMA_COLLECTION)

for chunk in chunks:
    embedding = get_embedding(chunk["text"])
    collection.add(
        ids=[str(chunk["chunk_id"])],
        documents=[chunk["text"]],
        embeddings=[embedding],
        metadatas=[{
            "team": chunk["team"],
            "paragraph_index": chunk["paragraph_index"]
        }]
    )

print(f"Chroma collection '{CHROMA_COLLECTION}' created with {len(chunks)} chunks.")


Chroma collection 'technical_docs' created with 74 chunks.


## Querying the RAG Pipeline

This section defines the functions used to query the RAG system. The `query_rag()` function retrieves the most relevant text chunks from the vector database based on a user query. First, the query is converted into an embedding using the same embedding model used for the document chunks. The vector database then performs a similarity search to find the top `k` most relevant chunks.

The retrieved chunks are combined into a context block that will be provided to the language model. A prompt is then constructed that instructs the model to answer the question **only using the retrieved context**, ensuring that responses remain grounded in the source documents.

The `ask_llm()` function sends this prompt to the language model, which generates the final answer. By combining retrieval from the vector database with a language model response, the system can answer questions using information extracted from the technical PDFs.

In [7]:
def query_rag(query: str, top_k: int = 3, collection_name=CHROMA_COLLECTION):
    collection = chroma_client.get_collection(name=collection_name)
    query_vec = get_embedding(query)
    results = collection.query(
        query_embeddings=[query_vec],
        n_results=top_k,
        include=["documents", "metadatas"]
    )

    retrieved_chunks = [
        {"text": doc, "metadata": meta}
        for doc, meta in zip(results["documents"][0], results["metadatas"][0])
    ]

    context = "\n\n".join([f"- {c['text']}" for c in retrieved_chunks])
    prompt = f"""
You are a technical documentation assistant. Answer the question ONLY using the context below.
If the answer is not in the context, respond: "I don't know".

Context:
{context}

Question:
{query}

Answer:
"""
    return prompt, retrieved_chunks

def ask_llm(prompt: str, model: str = "gpt-4", max_tokens: int = 500) -> str:
    response = client_openai.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens
    )
    return response.choices[0].message.content.strip()

## Testing the RAG Pipeline

In this final step, the complete RAG pipeline is tested using a sample query about **DORA metrics in DevOps**. The query is first passed to the `query_rag()` function, which retrieves the most relevant text chunks from the vector database based on semantic similarity.

The retrieved context is printed to verify which document fragments were selected. These chunks are then inserted into the prompt that will be sent to the language model. Finally, the `ask_llm()` function generates an answer using the retrieved context.

For transparency and debugging purposes, the notebook also prints the generated prompt, the metadata of the retrieved chunks, and the final response produced by the language model. This allows us to inspect how the retrieval step influences the generated answer.

In [8]:
test_query = "What are the DORA metrics and why are they important in DevOps?"
    
# Get prompt + retrieved chunks
prompt, used_chunks = query_rag(test_query)

print("\nRetrieved Context:")
for c in used_chunks:
    print("\n---")
    print(c["text"][:300])

# Ask GPT
answer = ask_llm(prompt)

print("=== Generated Prompt ===")
print(prompt[:1000] + "...\n")  # first 1000 chars
print("=== Retrieved Chunks Metadata ===")
for c in used_chunks:
    print(c["metadata"])
print("\n=== GPT Answer ===")
print(answer)


Retrieved Context:

---
Relevant metrics DevOps Research and Assessment (DORA) has developed a series of metrics which are intended to measure software development efficiency and reliability. These metrics include: Deployment Frequency: Time between code deployments. Mean Lead Time for Changes: Time between code commit and

---
In 2012, a report called "State of DevOps" was first published by Alanna Brown at Puppet Labs. As of 2014, the annual State of DevOps report was published by Nicole Forsgren, Gene Kim, Jez Humble and others. They stated that the adoption of DevOps was accelerating. Also in 2014, Lisa Crispin and Jan

---
36. Shahin, Mojtaba; Ali Babar, Muhammad; Zhu, Liming (2017). "Continuous Integration, Delivery and Deployment: A Systematic Review on Approaches, Tools, Challenges and Practices" ( elivery_and_Deployment_A_Systematic_Review_on_Approaches_Tools_Challenges_and_ Practices). IEEE Access. 5: 3909–3943. 
=== Generated Prompt ===

You are a technical documentation as

This example demonstrates how a Retrieval-Augmented Generation (RAG) pipeline can answer questions using information extracted from technical PDF documents.

In this test case, the model successfully identified and explained the DORA metrics, including Deployment Frequency, Lead Time for Changes, Change Failure Rate, Failed Deployment Recovery Time, and Reliability. The response was generated strictly from the retrieved context, showing how RAG improves factual accuracy and reduces hallucination by limiting the model to provided documents.

# Hallucination Test nº1 – Question Outside the Knowledge Base

In this section, we evaluate the system’s ability to avoid hallucination.

The test query asks about **Kubernetes**, which is not included in the retrieved PDF documents. Since the RAG pipeline is designed to answer questions only using the provided context, this test verifies whether the system correctly refrains from generating unsupported information.

In [10]:
print("\n==============================")
print("TEST 1: Question NOT in docs")
print("==============================")

query = "What is Kubernetes and how does it work?"

prompt, used_chunks = query_rag(query)
answer = ask_llm(prompt)

print("\nQuestion:", query)
print("\nRetrieved Chunks:")
for c in used_chunks:
    print("-", c["metadata"])

print("\nAnswer:")
print(answer)


TEST 1: Question NOT in docs

Question: What is Kubernetes and how does it work?

Retrieved Chunks:
- {'team': 'Docker_(software)', 'paragraph_index': 11}
- {'team': 'Docker_(software)', 'paragraph_index': 9}
- {'paragraph_index': 10, 'team': 'Docker_(software)'}

Answer:
I don't know.


In this test, the retriever returned Docker-related chunks due to semantic similarity with the Kubernetes query. Although the retrieval was not fully aligned with the question, the language model did not fabricate an answer and followed the prompt instructions. This demonstrates correct system behavior, as no hallucination occurred. Even when retrieval is imperfect, the LLM respected the constraint to rely only on the provided context.

# Hallucination Test nº2 – Partial Information Scenario

In this test, we evaluate how the RAG system handles questions where the retrieved context contains partial but relevant information.

The query asks who created the DORA metrics. The retriever searches the vector database and returns the most semantically similar chunks, which may include background information about DevOps Research and Assessment (DORA), related publications, or contributors.

The retrieved context is displayed to verify what information was selected. The model then generates an answer strictly based on the provided context. This test helps assess whether the system can correctly extract relevant facts when they are available in the documents, without introducing unsupported details.

This scenario demonstrates how RAG improves factual accuracy by grounding responses in retrieved content while still allowing the model to synthesize information from multiple chunks.

In [11]:
print("\n==============================")
print("TEST 2: Partial information")
print("==============================")

query = "Who created the DORA metrics?"

prompt, used_chunks = query_rag(query)

print("\nRetrieved Context:")
for c in used_chunks:
    print("\n---")
    print(c["text"][:300])

answer = ask_llm(prompt)

print("\nQuestion:", query)
print("\nRetrieved Chunks:")
for c in used_chunks:
    print("-", c["metadata"])

print("\nAnswer:")
print(answer)


TEST 2: Partial information

Retrieved Context:

---
Relevant metrics DevOps Research and Assessment (DORA) has developed a series of metrics which are intended to measure software development efficiency and reliability. These metrics include: Deployment Frequency: Time between code deployments. Mean Lead Time for Changes: Time between code commit and

---
In 2012, a report called "State of DevOps" was first published by Alanna Brown at Puppet Labs. As of 2014, the annual State of DevOps report was published by Nicole Forsgren, Gene Kim, Jez Humble and others. They stated that the adoption of DevOps was accelerating. Also in 2014, Lisa Crispin and Jan

---
19. Turner, Graham (20 November 2023). "Report: Software Engineers Face Backlash for Reporting Wrongdoing" ( reporting-wrongdoing/). DIGIT. Retrieved 5 January 2024. 20. Saran, Cliff. "Software engineers worry about speaking out - Computer Weekly" ( w.computerweekly.com/news/366560292/Software-engin

Question: Who created the DORA m

In this test, the system correctly retrieved relevant documents containing information about DevOps Research and Assessment (DORA). Although the retrieved context included additional background material, the language model produced a concise and accurate answer by identifying that the DORA metrics were created by DevOps Research and Assessment (DORA). The response did not introduce any unsupported information, demonstrating that the RAG pipeline successfully grounded the answer in the retrieved context and avoided hallucination.

## Hallucination Test – False Assumption Scenario

In this test, we evaluate how the RAG system handles a question that contains an incorrect assumption. The query asks about the “10 DORA metrics,” however, the retrieved documents indicate that DORA defines a smaller set of metrics.

The goal of this test is to verify whether the system avoids fabricating additional metrics that are not present in the retrieved context. After retrieving the most relevant chunks, the model generates an answer strictly based on the provided information.

This scenario is important because it tests the system’s ability to challenge false assumptions and prevent hallucination, even when the question implies information that does not exist in the documents.

In [12]:
print("\n==============================")
print("TEST 3: False assumption")
print("==============================")

query = "What are the 10 DORA metrics?"

prompt, used_chunks = query_rag(query)

print("\nRetrieved Context:")
for c in used_chunks:
    print("\n---")
    print(c["text"][:300])

answer = ask_llm(prompt)

print("\nQuestion:", query)
print("\nRetrieved Chunks:")
for c in used_chunks:
    print("-", c["metadata"])

print("\nAnswer:")
print(answer)


TEST 3: False assumption

Retrieved Context:

---
Relevant metrics DevOps Research and Assessment (DORA) has developed a series of metrics which are intended to measure software development efficiency and reliability. These metrics include: Deployment Frequency: Time between code deployments. Mean Lead Time for Changes: Time between code commit and

---
In 2012, a report called "State of DevOps" was first published by Alanna Brown at Puppet Labs. As of 2014, the annual State of DevOps report was published by Nicole Forsgren, Gene Kim, Jez Humble and others. They stated that the adoption of DevOps was accelerating. Also in 2014, Lisa Crispin and Jan

---
19. Turner, Graham (20 November 2023). "Report: Software Engineers Face Backlash for Reporting Wrongdoing" ( reporting-wrongdoing/). DIGIT. Retrieved 5 January 2024. 20. Saran, Cliff. "Software engineers worry about speaking out - Computer Weekly" ( w.computerweekly.com/news/366560292/Software-engin

Question: What are the 10 DORA metr

In this false-assumption test, the system correctly identified that the retrieved context does not contain information about 10 DORA metrics. Instead of fabricating an answer, the model responded appropriately by stating that the context does not provide the requested information.

# Conclusion

This project demonstrates the implementation of a Retrieval-Augmented Generation (RAG) pipeline for extracting and answering questions based on technical PDF documents. The system successfully performs document ingestion, text cleaning, paragraph chunking, embedding generation, vector storage, semantic retrieval, and context-aware answer generation.

The evaluation tests show that the pipeline behaves correctly in different scenarios. When the answer is available in the documents, the system retrieves relevant chunks and generates accurate responses. When information is partially available, the model synthesizes the retrieved content without introducing unsupported details. Most importantly, in hallucination tests and false-assumption scenarios, the system correctly refrains from fabricating information and responds only based on the provided context.

Overall, the results confirm that the RAG architecture improves factual reliability, reduces hallucinations, and enables AI systems to answer domain-specific questions using external knowledge sources. This makes the approach highly suitable for technical documentation assistants and knowledge-based applications.